## §0 — Setup

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import open3d as o3d
import pyvista as pv
from PIL import Image as PILImage

from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.mesh.tsdf import Open3DTSDFFusion
from collab_splats.mesh.utils import find_depth_edges, pointcloud_to_mesh
from collab_splats.utils.visualization import (
    visualize_splat, o3d_mesh_to_polydata,
    MESH_KWARGS, VIZ_KWARGS, CAMERA_KWARGS,
)

# Notebook lives at docs/source/tutorials/06_mesh/stage/
# Cache is at docs/source/.cache/birds_c0043/
CACHE_DIR  = Path("../../.cache/birds_c0043")
OMEGA_ZARR = CACHE_DIR / "omega" / "reconstruction.zarr"
OUTPUT_DIR = Path("/tmp/mesh_compare")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Omega zarr exists: {OMEGA_ZARR.exists()}")
print(f"Output dir: {OUTPUT_DIR}")

## §1 — Load Omega Result

In [ ]:
result = FeedforwardResult.load_zarr(OMEGA_ZARR)

print(f"points:        {result.points.shape}")
print(f"colors:        {result.colors.shape}")
print(f"extrinsics:    {result.extrinsics.shape}")
print(f"intrinsics:    {result.intrinsics.shape}")
print(f"model_size:    {result.model_height} x {result.model_width}")
print(f"world_points:  {result.world_points.shape if result.world_points is not None else None}")
print(f"depth:         {result.depth.shape if result.depth is not None else None}")
conf_np = result.confidence.numpy() if hasattr(result.confidence, 'numpy') else result.confidence
print(f"confidence:    {conf_np.shape if conf_np is not None else None}")
print(f"pixel_indices: {result.pixel_indices.shape if result.pixel_indices is not None else None}")
print(f"image_paths[0]: {result.image_paths[0]}")

## §2 — Inspect Depth + Confidence

In [ ]:
import matplotlib.pyplot as plt

depth = result.depth  # (N, H, W)
conf_np = result.confidence.numpy() if hasattr(result.confidence, 'numpy') else result.confidence

# World-space bounding box from result.points
pts = result.points
bbox_min, bbox_max = pts.min(0), pts.max(0)
extent = bbox_max - bbox_min
print(f"Scene bbox:  {np.round(bbox_min, 2)} → {np.round(bbox_max, 2)}")
print(f"Extent (m):  {np.round(extent, 2)}  (max={extent.max():.1f}m)")
print()
print(f"Depth range: [{depth.min():.3f}, {depth.max():.3f}] m")
print(f"Depth 95th pct: {np.percentile(depth, 95):.3f} m")
print(f"Conf  range: [{conf_np.min():.4f}, {conf_np.max():.4f}]")
print(f"Conf  median: {np.median(conf_np):.4f}")
print()

# Suggested outdoor TSDF params
_extent_max = float(extent.max())
_voxel = max(0.05, _extent_max / 200.0)
_trunc  = _voxel * 5.0
_dtrunc = float(np.percentile(depth, 99)) * 1.2
print(f"Suggested outdoor TSDF params:")
print(f"  voxel_size={_voxel:.3f}  sdf_trunc={_trunc:.3f}  depth_trunc={_dtrunc:.1f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(depth[0], cmap='plasma'); axes[0].set_title('Depth frame 0')
axes[1].imshow(conf_np[0], cmap='viridis'); axes[1].set_title('Confidence frame 0')
axes[2].hist(depth.ravel(), bins=100, log=True); axes[2].set_title('Depth distribution')
plt.tight_layout(); plt.show()

## §3 — Denoising Variants

Three variants compared:
- **A (raw):** `result.points` unmodified
- **B (tier-2):** conf + depth-edge + depth-trunc mask on `world_points` — mirrors VGGT-Omega's own viz
- **C (tier-3):** statistical outlier removal on `result.points` with aligned `pixel_indices`

In [ ]:
# ── Variant A: raw ────────────────────────────────────────────────────────────
pts_A    = result.points.copy()
colors_A = result.colors.copy()
print(f"A (raw):              {len(pts_A):,} points")

# ── Variant B: conf + depth-edge + depth-trunc mask ──────────────────────────
CONF_PERCENTILE_B = 30.0
DEPTH_TRUNC_B     = float(np.percentile(depth, 99)) * 1.1

conf_b = conf_np.copy()
for i in range(len(result.image_paths)):
    conf_b[i][find_depth_edges(depth[i])] = 0.0
conf_b[depth > DEPTH_TRUNC_B] = 0.0
valid_conf = conf_b[conf_b > 0]
thresh_b = np.percentile(valid_conf, CONF_PERCENTILE_B) if len(valid_conf) else 0.0
mask_b = conf_b > thresh_b  # (N, H, W)

N, H, W, _ = result.world_points.shape
imgs_b = []
for p in result.image_paths:
    img = PILImage.open(p).convert("RGB").resize((W, H), PILImage.BILINEAR)
    imgs_b.append(np.asarray(img, dtype=np.uint8))
imgs_b = np.stack(imgs_b)  # (N, H, W, 3)

pts_B    = result.world_points[mask_b].astype(np.float32)
colors_B = imgs_b[mask_b]
print(f"B (conf+edge+trunc):  {len(pts_B):,} points  (depth_trunc={DEPTH_TRUNC_B:.1f}m, conf_pct={CONF_PERCENTILE_B})")

# ── Variant C: statistical outlier removal ────────────────────────────────────
pcd_c = o3d.geometry.PointCloud()
pcd_c.points = o3d.utility.Vector3dVector(result.points)
pcd_c.colors = o3d.utility.Vector3dVector(result.colors.astype(np.float64) / 255.0)
_, ind_c = pcd_c.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
ind_c = np.asarray(ind_c, dtype=np.int64)

pts_C    = result.points[ind_c]
colors_C = result.colors[ind_c]
pix_C    = result.pixel_indices[ind_c] if result.pixel_indices is not None else None
print(f"C (stat outlier):     {len(pts_C):,} points  ({len(result.points) - len(pts_C):,} removed)")

## §4 — TSDF (outdoor-tuned params)

Default params (`voxel=0.02m, sdf_trunc=0.08m, depth_trunc=10m`) are indoor-calibrated.
At 20–100m outdoor scale, `sdf_trunc` << per-frame depth noise → SDF cancellation → empty mesh.
Scale params to actual scene extent from §2.

In [ ]:
_extent_max = float((result.points.max(0) - result.points.min(0)).max())
_voxel_size  = max(0.05, _extent_max / 200.0)
_sdf_trunc   = _voxel_size * 5.0
_depth_trunc = float(np.percentile(depth, 99)) * 1.2

print(f"Outdoor TSDF params:")
print(f"  voxel_size  = {_voxel_size:.3f} m")
print(f"  sdf_trunc   = {_sdf_trunc:.3f} m")
print(f"  depth_trunc = {_depth_trunc:.1f} m")

_tsdf_dir = OUTPUT_DIR / "tsdf_tuned"
mesh_result_tsdf = pointcloud_to_mesh(
    result, _tsdf_dir,
    method="open3d_tsdf",
    voxel_size=_voxel_size,
    sdf_trunc=_sdf_trunc,
    depth_trunc=_depth_trunc,
    clean_repair=False,
)
mesh_tsdf = o3d.io.read_triangle_mesh(str(mesh_result_tsdf.mesh_path))
print(f"TSDF mesh: {len(mesh_tsdf.vertices):,} verts, {len(mesh_tsdf.triangles):,} triangles")

## §5 — Image-grid Mesh (world_points, full grid)

Per-frame quads from the full N×H×W `world_points` grid.
Quad validity = all 4 corners pass conf + depth-edge + depth-trunc mask.
Comparable to MapAnything's `predictions_to_glb` approach.
Vertices are NOT exactly `result.points` — uses full dense grid.

In [ ]:
def _image_grid_from_world_points(result, conf_percentile=30.0,
                                   depth_trunc_m=None, edge_threshold=0.01):
    """Image-grid mesh from full world_points grid with conf+edge+trunc masking."""
    N, H, W, _ = result.world_points.shape
    dep = result.depth  # (N, H, W)
    conf = conf_np.copy()  # (N, H, W) — from §1

    if depth_trunc_m is None:
        depth_trunc_m = float(np.percentile(dep, 99)) * 1.1

    for i in range(N):
        conf[i][find_depth_edges(dep[i], threshold=edge_threshold)] = 0.0
    conf[dep > depth_trunc_m] = 0.0
    valid_conf = conf[conf > 0]
    threshold = np.percentile(valid_conf, conf_percentile) if len(valid_conf) else 0.0
    mask = conf > threshold  # (N, H, W) bool

    all_verts, all_colors, all_faces = [], [], []
    offset = 0
    for i in range(N):
        m = mask[i]  # (H, W)
        idx = np.full((H, W), -1, dtype=np.int64)
        valid_pixels = np.argwhere(m)  # (P, 2) [row, col]
        if len(valid_pixels) == 0:
            continue
        idx[valid_pixels[:, 0], valid_pixels[:, 1]] = np.arange(len(valid_pixels)) + offset

        all_verts.append(result.world_points[i][m].astype(np.float32))
        img = PILImage.open(result.image_paths[i]).convert("RGB").resize((W, H), PILImage.BILINEAR)
        rgb = np.asarray(img, dtype=np.uint8)
        all_colors.append(rgb[m])

        quad_mask = (
            (idx[:-1, :-1] >= 0) & (idx[1:, :-1] >= 0) &
            (idx[1:,  1:] >= 0) & (idx[:-1, 1:] >= 0)
        )
        v00 = idx[:-1, :-1][quad_mask]
        v10 = idx[1:,  :-1][quad_mask]
        v11 = idx[1:,   1:][quad_mask]
        v01 = idx[:-1,  1:][quad_mask]
        tris = np.concatenate([
            np.stack([v00, v10, v11], 1),
            np.stack([v00, v11, v01], 1),
        ])
        all_faces.append(tris)
        offset += len(valid_pixels)

    verts  = np.concatenate(all_verts)
    colors = np.concatenate(all_colors)
    faces  = np.concatenate(all_faces) if all_faces else np.zeros((0, 3), dtype=np.int64)

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices      = o3d.utility.Vector3dVector(verts.astype(np.float64))
    mesh.triangles     = o3d.utility.Vector3iVector(faces.astype(np.int32))
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors.astype(np.float64) / 255.0)
    return mesh

mesh_wpts = _image_grid_from_world_points(result, conf_percentile=30.0)
print(f"image-grid (world_points): {len(mesh_wpts.vertices):,} verts, {len(mesh_wpts.triangles):,} triangles")

## §6 — Image-grid Mesh (pixel_indices, pcd-aligned)

Vertices === `result.points`. Topology from `pixel_indices` lookup.
Quad validity = all 4 corners survived creator's conf filter AND no depth edge.
Preserves exact pcd↔mesh correspondence — feature lifting works on both.

If `pixel_indices` is None in this zarr, cell prints a warning and skips.

In [ ]:
def _image_grid_from_pixel_indices(result, edge_threshold=0.01):
    """Image-grid mesh where vertices == result.points. Topology from pixel_indices."""
    if result.pixel_indices is None:
        print("WARNING: pixel_indices not in zarr — skipping §6.")
        return None

    N = len(result.image_paths)
    H, W = result.model_height, result.model_width
    dep = result.depth  # (N, H, W)

    # pixel → point-index lookup; -1 = filtered out by creator
    pixel_to_point = np.full((N, H, W), -1, dtype=np.int64)
    fi = result.pixel_indices[:, 0]
    ri = result.pixel_indices[:, 1]
    ci = result.pixel_indices[:, 2]
    pixel_to_point[fi, ri, ci] = np.arange(len(result.points))

    all_faces = []
    for i in range(N):
        idx = pixel_to_point[i]  # (H, W)
        valid = idx >= 0

        if dep is not None:
            edge_mask = find_depth_edges(dep[i], threshold=edge_threshold)
            no_edge = ~(
                edge_mask[:-1, :-1] | edge_mask[1:, :-1] |
                edge_mask[1:,  1:] | edge_mask[:-1, 1:]
            )
        else:
            no_edge = np.ones((H - 1, W - 1), dtype=bool)

        quad_mask = (
            valid[:-1, :-1] & valid[1:, :-1] &
            valid[1:,  1:] & valid[:-1, 1:] & no_edge
        )
        v00 = idx[:-1, :-1][quad_mask]
        v10 = idx[1:,  :-1][quad_mask]
        v11 = idx[1:,   1:][quad_mask]
        v01 = idx[:-1,  1:][quad_mask]
        tris = np.concatenate([
            np.stack([v00, v10, v11], 1),
            np.stack([v00, v11, v01], 1),
        ])
        all_faces.append(tris)

    faces = np.concatenate(all_faces) if all_faces else np.zeros((0, 3), dtype=np.int64)

    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices      = o3d.utility.Vector3dVector(result.points.astype(np.float64))
    mesh.triangles     = o3d.utility.Vector3iVector(faces.astype(np.int32))
    mesh.vertex_colors = o3d.utility.Vector3dVector(
        result.colors.astype(np.float64) / 255.0)
    return mesh

mesh_pidx = _image_grid_from_pixel_indices(result)
if mesh_pidx is not None:
    print(f"image-grid (pixel_indices): {len(mesh_pidx.vertices):,} verts, {len(mesh_pidx.triangles):,} triangles")

## §7 — Visualize Each Method

Camera frustum scale adjusted to scene extent (outdoor scenes need larger scale than default 0.02).

In [ ]:
_scene_scale = float((result.points.max(0) - result.points.min(0)).max())
_cam_kwargs = {**CAMERA_KWARGS, "scale": _scene_scale * 0.02, "n_poses": 5}

def _viz(name, mesh_o3d):
    if mesh_o3d is None:
        print(f"{name}: skipped (None)")
        return
    pd = o3d_mesh_to_polydata(mesh_o3d)
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(f"  verts={len(mesh_o3d.vertices):,}  tris={len(mesh_o3d.triangles):,}")
    visualize_splat(
        pd,
        aligned_cameras=list(result.extrinsics),
        mesh_kwargs=MESH_KWARGS,
        camera_kwargs=_cam_kwargs,
        viz_kwargs=VIZ_KWARGS,
    )

_viz("§4 TSDF tuned", mesh_tsdf)
_viz("§5 image-grid (world_points)", mesh_wpts)
_viz("§6 image-grid (pixel_indices)", mesh_pidx)

## §8 — Summary

In [ ]:
def _stats(name, mesh):
    if mesh is None:
        print(f"{name:<40} SKIPPED")
        return
    print(f"{name:<40} verts={len(mesh.vertices):>8,}  tris={len(mesh.triangles):>8,}")

print(f"{'Method':<40} {'Vertices':>8}  {'Triangles':>8}")
print("-" * 65)
_stats("TSDF tuned", mesh_tsdf)
_stats("image-grid (world_points)", mesh_wpts)
_stats("image-grid (pixel_indices)", mesh_pidx)

print("""
Visual quality notes (fill in after inspection):
  TSDF tuned              : ___
  image-grid world_points : ___
  image-grid pixel_indices: ___

Winner to promote to ImageGridMesher: ___
""")